In [ ]:
from __future__ import annotations

%load_ext autoreload
%autoreload 2

from pathlib import Path
from dotenv import load_dotenv
from collections import defaultdict
from tqdm.notebook import tqdm


import logging
import pdf_logging as pf
import sections as sf
import embedding as eb
import os, json

from google import genai

try:
    load_dotenv()
    API_KEY = os.getenv("GOOGLE_API_KEY")
except Exception as e:
    print(f"Error loading environment variable(s): {e}")

PROJECT_ROOT = Path.cwd().parent
PDF_DIR = PROJECT_ROOT / "Data" / "financebench" / "pdfs"
JSON_DIR = PROJECT_ROOT / "Data" / "cache" / "pages"
CHROMA_DIR = PROJECT_ROOT / "Data" / "chroma"

logger = logging.getLogger("pypdf")
logger.setLevel(logging.WARNING)
logger.propagate = False # Prevent propagation to the root logger, to not get wierd log and console dupes

log_file = PROJECT_ROOT / "Data" / "logs" /"pypdf_warnings.json"
handler = logging.FileHandler(log_file)
handler.setFormatter(pf.JsonFormatter())
logger.addHandler(handler)

## I. Ingestion and chunking

In [3]:
from ingestion import load_pdfs
from datasets import load_dataset

load_dotenv()
meta = {
    r["doc_name"]: {"company_name": r["company"], "doc_type": r["doc_type"], "year": r["doc_period"]}
    for r in load_dataset("PatronusAI/financebench", split="train")
}
documents = load_pdfs(
    sorted(PDF_DIR.glob("*.pdf")),
    meta=meta,
    log_file=log_file,
    cache_dir=PROJECT_ROOT / "Data" / "cache" / "pages",
)
print(f"Loaded {len(documents)} pages.")
print(f"Preview:\n{documents[0].page_content[:200]}")

loading pdfs:   0%|          | 0/2 [00:00<?, ?it/s]

skipped INTEL_2023_8K_dated-2023-08-16.pdf: PdfStreamError: Stream has ended unexpectedly
skipped INTEL_2023_8K_dated-2023-02-10.pdf: PdfStreamError: Stream has ended unexpectedly


Loaded 53901 pages.
Preview:
Table of Contents
 
 
low
 
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
 
FORM 10-K
 
☒
   ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE
SECURITIES EXCHANGE ACT OF 1


## II. Sectionning and embedding

In [27]:
test_document = documents[0].metadata["doc_name"]

pages = sorted(
    (d for d in documents if d.metadata["doc_name"] == test_document),
    key = lambda d: d.metadata["page"]
)
form = sf.form_of(test_document, pages[0].metadata.get("doc_type"))
headings = sf.find_headings([p.page_content for p in pages], form)

print(test_document, form, len(pages), len(headings))

#-----------------------------------------------------

sections = sf.split_by_section(pages)
print(f"Split into {len(sections)} sections.")


3M_2015_10K 10-K 158 20
Split into 178 sections.


In [ ]:
embedding_client = genai.Client(api_key=API_KEY)
collection = eb.get_collection(CHROMA_DIR)

pieces = eb.prepare(sections)
print(f"{len(sections)} sections -> {len(pieces)} pieces")

# batch_size: lower it (10) if you get 429 RESOURCE_EXHAUSTED; finished batches are skipped on a rerun
n = eb.index(collection, embedding_client, pieces, batch_size=20)
print(f"embedded {n} new pieces, collection holds {collection.count()}")

In [ ]:
hits = eb.search(
    collection, embedding_client,
    "What were 3M's net sales in 2015?",
    k=5,
    where={"company_name": "3M"},
)
for h in hits:
    md = h["metadata"]
    print(f"{h['distance']:.3f}  p{md['page']} (label {md.get('page_label')})  {md.get('section_title')}  {h['text'][:80]!r}")